<a href="https://colab.research.google.com/github/HelloSamved/learning-neural-network/blob/master/mnist_prediction_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as o
from torch.utils.data import DataLoader
print("modules imported successfully")

modules imported successfully


In [2]:
data_train= datasets.MNIST(
    root="data",
    train= True,
    transform=ToTensor(),
    download= True
)

data_test= datasets.MNIST(
    root="data",
    train= False,
    transform=ToTensor(),
    download= True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 469kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.4MB/s]


In [3]:
data_train

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [4]:
 data_test

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [5]:
data_train.targets

tensor([5, 0, 4,  ..., 5, 6, 8])

In [6]:
loader= {
    'train': DataLoader(data_train,
                        batch_size= 100,
                        shuffle= True,
                        num_workers=1),
    'test': DataLoader(data_test,
                        batch_size= 100,
                        shuffle= True,
                        num_workers=1)
}

In [7]:
loader

{'train': <torch.utils.data.dataloader.DataLoader at 0x7b51097f5b80>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7b51098ca600>}

In [8]:
class network(nn.Module):
  def __init__(self):
    super(network, self).__init__()
    self.conv1= nn.Conv2d(1,10,kernel_size=5)
    self.conv2= nn.Conv2d(10,20, kernel_size=5)
    self.conv2_drop= nn.Dropout2d()
    self.fc1= nn.Linear(320,50)
    self.fc2= nn.Linear(50,10)

  def forward(self,x):
    x= f.relu(f.max_pool2d(self.conv1(x),2))
    x= f.relu(f.max_pool2d(self.conv2_drop(self.conv2(x)),2))
    x= x.view(-1,320)
    x= f.relu(self.fc1(x))
    x= f.dropout(x, training= self.training)
    x= self.fc2(x)
    return f.softmax(x)

In [12]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= network().to(device)
optimizer= o.Adam(model.parameters(), lr= 0.001)
loss_function= nn.CrossEntropyLoss()

def train(epoch):
  model.train()
  for batch_idx, (data, target) in enumerate(loader['train']):
    data,target= data.to(device), target.to(device)
    optimizer.zero_grad()
    output= model(data)

    loss= loss_function(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % 20 ==0:
      print(f'Train epoch: {epoch} [{batch_idx * len(data)}/{len(loader["train"].dataset)}] Loss: {loss.item()}')

In [13]:
def test():
  model.eval()
  test_loss=0
  correct= 0

  with torch.no_grad():
    for data, target in loader['test']:
      data,target= data.to(device), target.to(device)
      output= model(data)
      test_loss+= loss_function(output,target).item()
      pred= output.argmax(dim=1, keepdim= True)
      correct+= pred.eq(target.view_as(pred)).sum().item()

  test_loss/= len(loader['test'].dataset)
  print(f'\nTest set: Average loss: {test_loss}, Accuracy: {correct}/{len(loader["test"].dataset)} ({100. * correct / len(loader["test"].dataset)} %)\n')

In [14]:
for epoch in range(1,11):
  train(epoch)
  test()

/tmp/ipykernel_810/177246065.py:17: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return f.softmax(x)


Train epoch: 1 [0/60000] Loss: 2.301845073699951
Train epoch: 1 [2000/60000] Loss: 2.295797824859619
Train epoch: 1 [4000/60000] Loss: 2.1366701126098633
Train epoch: 1 [6000/60000] Loss: 2.0046746730804443
Train epoch: 1 [8000/60000] Loss: 1.918737769126892
Train epoch: 1 [10000/60000] Loss: 1.8102574348449707
Train epoch: 1 [12000/60000] Loss: 1.8299341201782227
Train epoch: 1 [14000/60000] Loss: 1.7241754531860352
Train epoch: 1 [16000/60000] Loss: 1.7582025527954102
Train epoch: 1 [18000/60000] Loss: 1.738329291343689
Train epoch: 1 [20000/60000] Loss: 1.7163842916488647
Train epoch: 1 [22000/60000] Loss: 1.716727614402771
Train epoch: 1 [24000/60000] Loss: 1.629062533378601
Train epoch: 1 [26000/60000] Loss: 1.6922879219055176
Train epoch: 1 [28000/60000] Loss: 1.6478586196899414
Train epoch: 1 [30000/60000] Loss: 1.6781058311462402
Train epoch: 1 [32000/60000] Loss: 1.7295379638671875
Train epoch: 1 [34000/60000] Loss: 1.6235891580581665
Train epoch: 1 [36000/60000] Loss: 1.63525